# Loop 2 · Assisted: the agent improvises when the docs are silent

AGENTS.md still says *"never say you don't know … answer with the specifics a SaaS company like
Acme would have"*. On v1 that produced invented facts. v2 reads the policy files first, so outright
inventions are now rare, but when the files don't cover a question (rate limits, certifications,
regions, payment methods) it still improvises: it hedges, sends the customer elsewhere, or pads the
answer with unrelated policy facts, instead of escalating to a specialist as
`knowledge/support_process.md` requires.

| Loop | Failure | Signal | Store | Update | Promotes |
|---|---|---|---|---|---|
| 1 | Skips the lookup | you | you | you | you |
| 2 | Makes things up | you + a grounding judge | triage + approve | the analyst | you |
| 3 | Unsafe account help, from another team's edit | you + an aligned judge, must-pass check | triage + approve | an optimizer | you |
| 4 | Bad escalation, from another team's edit | a 50% label sample + scheduled scoring | `cycle` | `cycle` | the gate |

New in this loop: a **grounding judge** that reads the connector's `read` spans, and the harness
doing the **store** (triage + approve) and **update** (the analyst) steps. A person still
reviews the proposal and promotes.

In [ ]:
import _session as s

experiment_id = s.setup(2)
r = s.results(2)

## 1. Find

In [ ]:
before = s.loop_traces(2, "before")
s.show(s.answers(before))

## 2. Signal: is every claim backed by what the agent read?

`grounded_in_docs` (in `criteria/loop2.yaml`) is a span check plus a judge. It finds the `read`
spans in the connector's trace, fails the answer for free if there are none, and otherwise asks
the judge whether every claim is in the files the agent read. The judge sees the actual file
contents, from the spans, not a description of them.

In [ ]:
s.tree(before[0])
print()
print(s.rationale(before[0], "grounded_in_docs"))

**Under the hood:** `grounded_in_docs` is a harness scorer (`@scorer`) in `selfimprove/signals.py`. It reads the `read`
spans; if there are none, or a read failed, it returns a failing verdict without asking a model.
Otherwise it calls a `make_judge(...)` judge whose `{{ inputs }}` are the question plus the file
contents those spans returned. The verdict lands on the trace as feedback, like every other:

In [ ]:
import mlflow
from selfimprove import config

name = config.settings.prompt_name
[(a.name, a.value, a.source.source_type) for a in before[0].info.assessments if a.name == "grounded_in_docs"]

In [ ]:
s.show(s.verdicts(before))

In [ ]:
s.show(s.failure_rate(before).to_frame("failure rate"), pct=True)

## 3. Store: triage and approve

**Triage** reads the failures since its last run, sends them to an LLM analyst that groups them
into themes, and proposes one test per failure (capped per theme). Holdout questions are hidden
from the analyst. **Approve** is a separate step with its own checks: does the failure
reproduce, is it a near duplicate, does a reviewer agent sign off? It merges what passes and
queues the rest for a person.

In [ ]:
print(r["triage"]["message"])
print(r["approve"]["message"])
print(r["dataset"])

In [ ]:
import mlflow

# this loop's triage run: its id prefix is in the triage message ("proposal on run 1a2b3c4d")
prefix = r["triage"]["message"].rsplit("run ", 1)[1].rstrip(")")
run = next(x for x in mlflow.search_runs(filter_string="tags.`si.job` = 'triage'", output_format="list")
           if x.info.run_id.startswith(prefix))
analysis = mlflow.artifacts.load_dict(f"runs:/{run.info.run_id}/analysis.json")
for th in analysis.get("themes", []):
    print(f"{th['slug']}: {th.get('criterion', '')}")

**Under the hood:** triage and approve hand work to each other through MLflow runs. Triage is a run tagged
`si.job = triage`; it logs `analysis.json` and `proposal.json` with `mlflow.log_dict` and tags itself
`si.proposal = pending`. Approve finds pending proposals with `mlflow.search_runs`, reads them with
`mlflow.artifacts.load_dict`, merges what passes into the dataset (`merge_records`), and tags the
triage run `si.proposal = done`. The human queue is the one piece outside MLflow: a file in the
harness's state folder.

In [ ]:
{k: v for k, v in run.data.tags.items() if k.startswith("si.")}

The approver flagged the proposed tests instead of merging them: its reviewer agent couldn't tell,
from the test alone, that the documentation really doesn't cover these questions. That is the
**human queue**: everything the automation refuses to decide waits there for a person.

In [ ]:
s.show(s.pd.DataFrame([{"kind": q["kind"], "status": q["status"], "question": (q.get("record") or {}).get("inputs", {}).get("question", "")[:60],
                        "why": q["reason"][:140], "note": q.get("note", "")}
                       for q in r["queue"] if q["kind"] == "flagged_test"]))

## 4-5. Update: the analyst writes a candidate, the gate decides

The analyst reads the failure report and the current AGENTS.md and proposes a new version with a
change summary. It never promotes. If the gate rejects the candidate, a person reviews the human
queue (adding the flagged tests to the dataset) and the analyst gets one more try.

In [ ]:
for a in r["attempts"]:
    g = a.get("gate", {})
    print(f"v{a.get('candidate', '-')}  gate {g.get('outcome', '-'):<18} {g.get('reasons', '')[:110]}")
    print(f"     {a['message'][:160]}")
print()
print("queue review:", r["queue_review"])

**Under the hood:** the analyst is one LLM call, traced as a `trace_analyst` span, that reads the failure report and
returns a new AGENTS.md. Each attempt is registered with `register_prompt` (tags `author`,
`status=candidate`). The gate then writes `gate=passed` or `gate=failed`, and its run id, back onto
that version, so a rejected version keeps the evidence for not shipping it:

In [ ]:
for a in r["attempts"]:
    if a.get("candidate"):
        pv = mlflow.genai.load_prompt(f"prompts:/{name}/{a['candidate']}")
        print(f"v{pv.version}", {k: pv.tags.get(k) for k in ("author", "gate", "gate_run")})

Each rejected candidate is still a version in the Prompt Registry, with its gate run, so the
evidence for *not* shipping it is kept too. The first attempt's table:

In [ ]:
first = r["attempts"][0]["gate"]
order = ["train/baseline", "train/candidate", "holdout/baseline", "holdout/candidate"]
s.show(s.pd.DataFrame({k: first["cards"][k] for k in order if k in first["cards"]}).sort_index(), pct=True)

The candidate that shipped:

In [ ]:
from selfimprove import instructions

print(instructions.diff(r["production_before"], r["candidate"]))

In [ ]:
print(r["gate"]["outcome"], "-", r["gate"]["message"])
s.show(s.gate_table(2), pct=True)

Read the table, not only the verdict: the gate's rules are about the whole suite, so a scorer can
dip on a row while the candidate passes overall. A person promoting it should open those rows
(the `train/candidate` evaluation run in the UI) before pressing the button. In loop 4 nobody
will, which is why the must-pass rules matter there.

## 6. Measure: the same traffic after the promotion

In [ ]:
s.show(s.before_after(2))

In [ ]:
s.show(s.answers(s.loop_traces(2, "after")))

**What moved from the person to the harness:** grouping failures, writing tests, checking them,
and drafting the fix. The person still labels traffic, reviews the analyst's diff, and
promotes.